In [1]:
import pandas as pd
df = pd.read_csv("../data/processed/clean_matches.csv")
df.head()

,MatchDate,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway,FTResult
0,2015-08-08,Bournemouth,Aston Villa,1597.08,1580.38,7.0,13.0,3.0,6.0,0.0,1.0,A
1,2015-08-08,Chelsea,Swansea,1893.44,1673.08,4.0,10.0,3.0,9.0,2.0,2.0,D
2,2015-08-08,Everton,Watford,1709.04,1576.56,3.0,6.0,7.0,13.0,2.0,2.0,D
3,2015-08-08,Leicester,Sunderland,1636.10,1607.61,7.0,10.0,2.0,8.0,4.0,2.0,H
4,2015-08-08,Man United,Tottenham,1812.81,1733.16,5.0,5.0,6.0,7.0,1.0,0.0,H


# Feature engineering

In [2]:
team_history = df[["MatchDate", "HomeTeam", "AwayTeam", "FTHome", "FTAway"]].copy()

home = team_history[["MatchDate", "HomeTeam", "FTHome", "FTAway"]].copy()
home.columns = ["MatchDate", "Team", "GoalsScored", "GoalsConceded"]

away = team_history[["MatchDate", "AwayTeam", "FTHome", "FTAway"]].copy()
away.columns = ["MatchDate", "Team", "GoalsConceded", "GoalsScored"]

team_history = pd.concat([home, away])
team_history = team_history.sort_values(["Team", "MatchDate"])
team_history.head(10)

,MatchDate,Team,GoalsScored,GoalsConceded
6,2015-08-09,Arsenal,0.0,2.0
17,2015-08-16,Arsenal,2.0,1.0
29,2015-08-24,Arsenal,0.0,0.0
35,2015-08-29,Arsenal,1.0,0.0
40,2015-09-12,Arsenal,2.0,0.0
52,2015-09-19,Arsenal,0.0,2.0
60,2015-09-26,Arsenal,5.0,2.0
77,2015-10-04,Arsenal,3.0,0.0
86,2015-10-17,Arsenal,3.0,0.0
90,2015-10-24,Arsenal,2.0,1.0


In [3]:
team_history["AvgGoalsScored5"] = (
    team_history
    .groupby("Team")["GoalsScored"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_history["AvgGoalsConceded5"] = (
    team_history
    .groupby("Team")["GoalsConceded"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_history.head(20)

,MatchDate,Team,GoalsScored,GoalsConceded,AvgGoalsScored5,AvgGoalsConceded5
6,2015-08-09,Arsenal,0.0,2.0,NaN,NaN
17,2015-08-16,Arsenal,2.0,1.0,NaN,NaN
29,2015-08-24,Arsenal,0.0,0.0,NaN,NaN
35,2015-08-29,Arsenal,1.0,0.0,NaN,NaN
40,2015-09-12,Arsenal,2.0,0.0,NaN,NaN
52,2015-09-19,Arsenal,0.0,2.0,1.0,0.6
60,2015-09-26,Arsenal,5.0,2.0,1.0,0.6
77,2015-10-04,Arsenal,3.0,0.0,1.6,0.8
86,2015-10-17,Arsenal,3.0,0.0,2.2,0.8
90,2015-10-24,Arsenal,2.0,1.0,2.6,0.8


In [4]:
home_stats = team_history[["MatchDate", "Team", "AvgGoalsScored5", "AvgGoalsConceded5"]].copy()
home_stats = home_stats.rename(columns={
    "Team": "HomeTeam",
    "AvgGoalsScored5": "HomeAvgGoalsScored5",
    "AvgGoalsConceded5": "HomeAvgGoalsConceded5"
})

away_stats = team_history[[
    "MatchDate",
    "Team",
    "AvgGoalsScored5",
    "AvgGoalsConceded5"
]].copy()
away_stats = away_stats.rename(columns={
    "Team": "AwayTeam",
    "AvgGoalsScored5": "AwayAvgGoalsScored5",
    "AvgGoalsConceded5": "AwayAvgGoalsConceded5"
})

df = df.merge(
    home_stats,
    on=["MatchDate", "HomeTeam"],
    how="left"
)

df = df.merge(
    away_stats,
    on=["MatchDate", "AwayTeam"],
    how="left"
)


In [5]:
df.head(60)

,MatchDate,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway,FTResult,HomeAvgGoalsScored5,HomeAvgGoalsConceded5,AwayAvgGoalsScored5,AwayAvgGoalsConceded5
0,2015-08-08,Bournemouth,Aston Villa,1597.08,1580.38,7.0,13.0,3.0,6.0,0.0,1.0,A,NaN,NaN,NaN,NaN
1,2015-08-08,Chelsea,Swansea,1893.44,1673.08,4.0,10.0,3.0,9.0,2.0,2.0,D,NaN,NaN,NaN,NaN
2,2015-08-08,Everton,Watford,1709.04,1576.56,3.0,6.0,7.0,13.0,2.0,2.0,D,NaN,NaN,NaN,NaN
3,2015-08-08,Leicester,Sunderland,1636.10,1607.61,7.0,10.0,2.0,8.0,4.0,2.0,H,NaN,NaN,NaN,NaN
4,2015-08-08,Man United,Tottenham,1812.81,1733.16,5.0,5.0,6.0,7.0,1.0,0.0,H,NaN,NaN,NaN,NaN
5,2015-08-08,Norwich,Crystal Palace,1617.68,1649.26,4.0,10.0,6.0,6.0,1.0,3.0,A,NaN,NaN,NaN,NaN
6,2015-08-09,Arsenal,West Ham,1853.70,1609.12,5.0,8.0,0.0,4.0,0.0,2.0,A,NaN,NaN,NaN,NaN
7,2015-08-09,Newcastle,Southampton,1590.10,1717.73,4.0,4.0,3.0,4.0,2.0,2.0,D,NaN,NaN,NaN,NaN
8,2015-08-09,Stoke,Liverpool,1700.78,1755.30,7.0,8.0,1.0,4.0,0.0,1.0,A,NaN,NaN,NaN,NaN
9,2015-08-10,West Brom,Man City,1638.09,1884.95,4.0,8.0,9.0,15.0,0.0,3.0,A,NaN,NaN,NaN,NaN


In [6]:
df = df.dropna(subset=[
    "HomeAvgGoalsScored5",
    "HomeAvgGoalsConceded5",
    "AwayAvgGoalsScored5",
    "AwayAvgGoalsConceded5"
])
df.head()

,MatchDate,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway,FTResult,HomeAvgGoalsScored5,HomeAvgGoalsConceded5,AwayAvgGoalsScored5,AwayAvgGoalsConceded5
50,2015-09-19,Aston Villa,West Brom,1569.08,1634.21,1.0,4.0,4.0,5.0,0.0,1.0,A,1.2,1.6,0.6,1.2
51,2015-09-19,Bournemouth,Sunderland,1579.70,1577.37,4.0,4.0,2.0,2.0,2.0,0.0,H,1.2,1.8,1.2,2.2
52,2015-09-19,Chelsea,Arsenal,1843.40,1837.04,3.0,4.0,7.0,10.0,2.0,0.0,H,1.4,2.4,1.0,0.6
53,2015-09-19,Man City,West Ham,1908.94,1627.77,9.0,15.0,6.0,9.0,1.0,2.0,A,2.2,0.0,2.2,1.2
54,2015-09-19,Newcastle,Watford,1574.63,1585.95,1.0,2.0,4.0,6.0,1.0,2.0,A,0.4,1.4,0.6,0.8


In [7]:
df = df.drop(columns=["FTHome", "FTAway", "MatchDate", "HomeTeam", "AwayTeam"])
df.head()

,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTResult,HomeAvgGoalsScored5,HomeAvgGoalsConceded5,AwayAvgGoalsScored5,AwayAvgGoalsConceded5
50,1569.08,1634.21,1.0,4.0,4.0,5.0,A,1.2,1.6,0.6,1.2
51,1579.70,1577.37,4.0,4.0,2.0,2.0,H,1.2,1.8,1.2,2.2
52,1843.40,1837.04,3.0,4.0,7.0,10.0,H,1.4,2.4,1.0,0.6
53,1908.94,1627.77,9.0,15.0,6.0,9.0,A,2.2,0.0,2.2,1.2
54,1574.63,1585.95,1.0,2.0,4.0,6.0,A,0.4,1.4,0.6,0.8


In [8]:
df["EloDifference"] = df["HomeElo"] - df["AwayElo"]
df["FormDifference"] = df["Form5Home"] - df["Form5Away"]
df["GoalsScoredDifference5"] = (
    df["HomeAvgGoalsScored5"] -
    df["AwayAvgGoalsScored5"]
)

df["GoalsConcededDifference5"] = (
    df["HomeAvgGoalsConceded5"] -
    df["AwayAvgGoalsConceded5"]
)
df.head()

,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTResult,HomeAvgGoalsScored5,HomeAvgGoalsConceded5,AwayAvgGoalsScored5,AwayAvgGoalsConceded5,EloDifference,FormDifference,GoalsScoredDifference5,GoalsConcededDifference5
50,1569.08,1634.21,1.0,4.0,4.0,5.0,A,1.2,1.6,0.6,1.2,-65.13,-1.0,0.6,0.4
51,1579.70,1577.37,4.0,4.0,2.0,2.0,H,1.2,1.8,1.2,2.2,2.33,2.0,0.0,-0.4
52,1843.40,1837.04,3.0,4.0,7.0,10.0,H,1.4,2.4,1.0,0.6,6.36,-6.0,0.4,1.8
53,1908.94,1627.77,9.0,15.0,6.0,9.0,A,2.2,0.0,2.2,1.2,281.17,6.0,0.0,-1.2
54,1574.63,1585.95,1.0,2.0,4.0,6.0,A,0.4,1.4,0.6,0.8,-11.32,-4.0,-0.2,0.6


In [25]:
x = df.drop("FTResult", axis=1)
y = df["FTResult"]

In [10]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

# Splitting the data

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, shuffle=False)

# Normalising the data

In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [13]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(2943, 14)
(736, 14)
(2943,)
(736,)


# Train the model

In [14]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [15]:
y_pred = model.predict(X_test)

# Evaluation of the model

In [16]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_pred, y_test)
print(accuracy)

0.49592391304347827


In [17]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.52      0.57      0.55       248
           1       0.20      0.19      0.20       168
           2       0.62      0.60      0.61       320

    accuracy                           0.50       736
   macro avg       0.45      0.45      0.45       736
weighted avg       0.49      0.50      0.49       736



In [18]:
probabilities = model.predict_proba(X_test)

print(probabilities[0])

[0.46701378 0.32424207 0.20874415]


In [19]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[142  59  47]
 [ 67  32  69]
 [ 62  67 191]]


In [20]:
df["FTResult"].value_counts(normalize=True)

FTResult
H    0.447404
A    0.321555
D    0.231041
Name: proportion, dtype: float64

## Testing another model

In [23]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [24]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.54      0.55      0.55       248
           1       0.14      0.03      0.05       168
           2       0.55      0.77      0.64       320

    accuracy                           0.53       736
   macro avg       0.41      0.45      0.41       736
weighted avg       0.45      0.53      0.47       736

[[137  12  99]
 [ 60   5 103]
 [ 56  19 245]]


In [26]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_pred, y_test)
print(accuracy)

0.5258152173913043
